In [1]:
import os
import sys
sys.path.insert(0, "/Users/c_yang/Library/CloudStorage/OneDrive-Personal/nus/project/1_RuO2_stability/src")

import numpy as np
from matplotlib import pyplot as plt
from ase.visualize import view
from build_model import load_bulk_db, build_surface_model, build_interface_model, make_simple_supercell
from quick_equilibrium import quick_equilibrium

plt.style.use("acs")

## Load bulks from database

In [2]:
bulks = load_bulk_db("../working/dataset/RuO2_bulk_opt.db")
print(list(bulks.keys()))

['Ru-Hcp-r2scan', 'RuO2-SiF4-rpbe', 'RuO2-Fluorite-pbe', 'RuO2-SiF4-pbe', 'Ru-Fcc-r2scan', 'Ru-Fcc-pbe', 'RuO2-Fluorite-r2scan', 'Ru-Hcp-pbe', 'RuO2-Rutile-rpbe', 'RuO2-SiF4-r2scan', 'RuO2-Rutile-r2scan', 'Ru-Hcp-rpbe', 'RuO2-Rutile-pbe', 'RuO2-Cubic-rpbe', 'RuO2-Fluorite-rpbe', 'RuO2-Cubic-r2scan', 'RuO2-Cubic-pbe', 'Ru-Fcc-rpbe']


## Build surface model based on given parameters

In [3]:
bulk_info = "RuO2-Rutile-pbe"
miller_index = [0, 1, 1]
layers = 6 # Layer of Ru-O units
vacuum = 30 # Angstrom
symmetry = True # Whether to preserve symmetry when cleaving the surface
terminations = None #["Ru", "O"] # For "Ru" or "O" termination, it returns None
min_lattice = 12 # Minimum lattice vector length in Angstrom

In [4]:
surface = build_surface_model(bulks[bulk_info], miller_index, layers, vacuum, terminations)
if len(surface) == 0:
    print("No valid surface found.")
else:
    print(f"Number of surfaces found: {len(surface)}")

Number of surfaces found: 2


In [5]:
super_surf = []
for surf in surface:
    print(f"Surface info: {surf.cell.cellpar()}, {surf.get_chemical_formula()}")
    super_surf.append(make_simple_supercell(surf, min_lattice))

Surface info: [ 3.1242727   6.39392899 49.18153542 90.02066997 89.99998998 89.99999631], O24Ru12
Surface info: [ 3.1242727   6.39392899 49.18153542 90.02066997 89.99998998 89.99999631], O24Ru12


## Build interface model based on given parameters

### Neutral water box

In [6]:
interface = build_interface_model(surface = super_surf[0],
                      solvation="H2O",
                      pH = 7,
                      verbose=False,
                      surface_height=1.0
                      )

The number of solvent molecules is: 166
Interface model with 166 water molecules and extra species  generated.


In [7]:
view(interface, viewer="x3d")

### Netural water box with ions

In [8]:
interface = build_interface_model(surface = super_surf[0],
                      solvation="H2O",
                      pH = 7,
                      verbose=False,
                      ions=["Cl-", "Na+"],
                      ions_number=[1, 1],
                      surface_height=1.0
                      )

The number of solvent molecules is: 166
Interface model with 166 water molecules and extra species Cl,Na generated.


In [9]:
view(interface, viewer="x3d")
# view(interface)

### Netural water box with ions at the middle of the box

In [10]:
interface = build_interface_model(surface = super_surf[0],
                      solvation="H2O",
                      pH = 7,
                      verbose=False,
                      ions=["Cl-", "Na+"],
                      ions_number=[1, 1],
                      surface_height=1.0,
                      region = "middle"
                      )


The number of solvent molecules is: 166
Interface model with 166 water molecules and extra species Cl,Na generated.


In [11]:
view(interface, viewer="x3d")

### Acid water box with a Cl- ion

In [12]:
interface = build_interface_model(surface = super_surf[0],
                      solvation="H2O",
                      pH = 0,
                      ions = ["Cl-"],
                      ions_number = [1],
                      verbose=False,
                      surface_height=1.0,
                      region="bottom"
                      )

The number of solvent molecules is: 166
Interface model with 166 water molecules and extra species Cl,H3O generated.


In [13]:
view(interface, viewer="x3d")

### Alkaline water box with a K+ ion

In [14]:
interface = build_interface_model(surface = super_surf[0],
                      solvation="H2O",
                      pH = 14,
                      ions = ["K+"],
                      ions_number = [1],
                      verbose=False,
                      surface_height=1.0,
                      region="top"
                      )

The number of solvent molecules is: 166
Interface model with 166 water molecules and extra species K,HO generated.


In [15]:
view(interface, viewer="x3d")

## Extra bulk model

### Cu bulk

In [16]:
from ase.build import bulk

Cu = bulk("Cu", "fcc", a=3.6)


### Cu surface

In [17]:
miller_index = [1, 1, 1]
layers = 1 # Layer of Ru-O units
vacuum = 30 # Angstrom
symmetry = True # Whether to preserve symmetry when cleaving the surface
terminations = None #["Ru", "O"] # For "Ru" or "O" termination, it returns None
min_lattice = 12 # Minimum lattice vector length in Angstrom

In [18]:
surface = build_surface_model(Cu, miller_index, layers, vacuum, symmetry)
if len(surface) == 0:
    print("No valid surface found.")
else:
    print(f"Number of surfaces found: {len(surface)}")
super_surf = []
for surf in surface:
    print(f"Surface info: {surf.cell.cellpar()}, {surf.get_chemical_formula()}")
    super_surf.append(make_simple_supercell(surf, min_lattice))

Number of surfaces found: 1
Surface info: [ 2.54558441  2.54558441 36.23538291 90.         90.         60.        ], Cu3


In [19]:
view(super_surf[0], viewer="x3d")

### Build interface with Cu(111) surface

In [20]:
interface = build_interface_model(surface = super_surf[0],
                      solvation="H2O",
                      pH = 14,
                      ions = ["K+", "Cl-"],
                      ions_number = [1, 1],
                      verbose=False,
                      surface_height=1.0,
                      region="top"
                      )

The number of solvent molecules is: 150
Interface model with 150 water molecules and extra species K,Cl generated.


In [21]:
view(interface, viewer="x3d")

## Graphene

### Generate graphene

In [22]:
from ase.build.ribbon import graphene_nanoribbon
graphene = graphene_nanoribbon(6, 6, type="zigzag", saturated=False, vacuum=15.0, sheet=True)

def redefine_cell(atoms, scale_positions=False):
    matrix = np.asarray(
        [[1, 0, 0],
         [0, 0, 1],
         [0, 1, 0]]
    )
    atoms_cell =  matrix @ atoms.get_cell() @ matrix
    atoms.set_cell(atoms_cell)
    if scale_positions:
        atoms_positions = matrix @ atoms.get_positions().T
        atoms.set_positions(atoms_positions.T)
    print(atoms_cell)
    return atoms
graphene = redefine_cell(graphene, scale_positions=True)
graphene.positions[:, 2] -= 15.0

[[12.78        0.          0.        ]
 [ 0.         14.75707288  0.        ]
 [ 0.          0.         30.        ]]


In [23]:
view(graphene, viewer="x3d")

### Alkaline water box with a K+ ion

In [24]:
interface = build_interface_model(surface = graphene,
                      solvation="H2O",
                      pH = 14,
                      ions = ["K+"],
                      ions_number = [2],
                      sol_height=10.0,
                      verbose=False,
                      surface_height=1.0,
                      region="top"
                      )
view(interface, viewer="x3d")

The number of solvent molecules is: 189
Interface model with 63 water molecules and extra species K,HO generated.


## Quick equilibrium of the interface model by uMLIP

In [25]:
os.environ["MACE_MLIP"] = "/Users/c_yang/Documents/play_with_code/benchmarking/mace-mpa-0-medium.model"
atoms_eq = quick_equilibrium(interface, model="mace", nsteps=10, timestep=1, temperature=300, ensemble="NVT")

/Users/c_yang/Library/CloudStorage/OneDrive-Personal/nus/project/1_RuO2_stability/.venv/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.
Using head default out of ['default']
Default dtype float32 does not match model dtype float64, converting models to float32.


/Users/c_yang/Library/CloudStorage/OneDrive-Personal/nus/project/1_RuO2_stability/.venv/lib/python3.12/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


In [26]:
view(atoms_eq, viewer="x3d")